# 用 MCMC 样本直接重建 M、S、v（全部 K×K）

旧 notebook 为了讲原理，习惯性地枚举了全部 16 个构型、用 **完整 H（16×16）** 显式构造
$M=\sum_x \psi^\dagger H\psi$、$S=\sum_x \psi^\dagger\psi$。这造成一个错觉：好像"转回本征态"要先对角线化 16×16 的 H，成本连精确对角化都不如。

本 notebook 彻底改成**基于 MCMC 样本**的重建，全程只出现 **K×K** 矩阵：

- `M` : K×K（哈密顿量在 K 列子空间上的投影）
- `S` : K×K（K 列的重叠）
- `v` : K×K（广义本征问题的旋转系数）

关键恒等式（每个 walker 都成立）：
$$
\big(P^\dagger H P,\; P^\dagger P\big)_{\text{K×K 广义本征}}
\Longleftrightarrow
\underbrace{P^{-1} H P}_{\text{训练的 }E_L\text{ 矩阵，K×K}},\quad \text{本征值相等}
$$

$$
(P^\dagger P)^{-1}(P^\dagger H P) = P^{-1}(P^{\dagger})^{-1} P^\dagger H P = P^{-1} H P
$$

其中单 walker 矩阵 $P_{\alpha\beta}=\psi_\beta(x_\alpha)$（K×K，把 $\beta$ 列拟设算到 $\alpha$ 个副本构型上），$H\psi$ 用训练的局部能量算子 $\texttt{Ham\_Psi\_scaled}$ 作用在**被采样的构型**上（不触碰完整 H 矩阵）。


## 0. 载入环境与训练历史

In [8]:
import pickle
import numpy as np
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
from scipy.linalg import eigh

from NES_VMC_V1 import (
    NESTotalAnsatz_stable,
    create_single_machine_gauge_fixed,
    Ham_Psi_scaled,
)
from H2_631G import SINGLE_SIZE, ha, hi, K, Hatree_Fock, E_fcis, hi_ext

HISTORY_FILE = './data/26-09-10-16-25_history_natural_gradient_H2_molecule_K4.pkl'
with open(HISTORY_FILE, 'rb') as f:
    history = pickle.load(f)

print('history keys:', list(history.keys()))
print(f'K = {K}, 单态构型数 = {hi.n_states}, FCI 能级 = {np.round(E_fcis, 5)}')
print(f'samples shape = {history["samples"][-1].shape}')

params = history['params'][-1]
samples = np.asarray(history['samples'][-1])          # (N, K*SINGLE)，整数构型
N = samples.shape[0]
x_batch = samples.reshape(N, K, SINGLE_SIZE)           # (N, K, 8)，保留整数给算符/machine
print(f'重排后 walker 构型 x_batch.shape = {x_batch.shape}（N, K, n_spin）')

history keys: ['steps', 'logpsi_mean', 'logpsi_min', 'logpsi_max', 'grad_norm_raw', 'grad_norm_natural', 'E_L_real', 'E_L_imag', 'loss', 'first_step', 'save_interval', 'Energy_levels', 'params', 'samples']
K = 4, 单态构型数 = 16, FCI 能级 = [-1.05435 -0.95791 -0.66895 -0.5514 ]
samples shape = (3200, 32)
重排后 walker 构型 x_batch.shape = (3200, 4, 8)（N, K, n_spin）


## 1. 重建单 walker 的 K×K 矩阵 $P$ 与 $H\psi$

逐列拟设作用到已采样的副本构型上：$P_{n\alpha\beta}=\psi_\beta(x_{n\alpha})$，
并用 $(H\psi)_{n\alpha\beta}=(H\psi_\beta)(x_{n\alpha})$ 由求和路径弹算 $(H\psi)$（不构造 16×16 矩阵）。

In [9]:
# 用与训练同构的总拟设拿 graphdef，并构造列级 gauge-fixed machine（与训练一致）
total_ansatz = NESTotalAnsatz_stable(
    n_spin_orbitals=SINGLE_SIZE, n_states=K, hidden_dim=SINGLE_SIZE + K,
    rngs=nnx.Rngs(11),
)
single_machine_list = [
    create_single_machine_gauge_fixed(ansatz, Hatree_Fock)[0]
    for ansatz in total_ansatz.single_ansatz_list
]

x_jax = jnp.asarray(x_batch)
# P_{nαβ} = ψ_β(x_{nα})：第 β 列 macheine 对 (N,K,8) 求值 -> (N,K)
logP_cols = [
    single_machine_list[j](params['single_ansatz_list'][j], x_jax)
    for j in range(K)
]
logP = jnp.stack(logP_cols, axis=-1)                 # (N, α, β)

# 每 walker 的全局安全 shift（只做数值稳定；作为每 walker 的公共标度会在归一化里抵消）
shift_n = jnp.max(logP.real, axis=(1, 2)).reshape(-1)
P = jnp.exp(logP - shift_n.reshape(-1, 1, 1))         # (N, K, K)

HP = Ham_Psi_scaled(
    ha=ha,
    single_machine_list=single_machine_list,
    total_params=params,
    x=x_jax,
    shift=shift_n,
)

finite = (
    jnp.all(jnp.isfinite(P), axis=(-2, -1))
    & jnp.all(jnp.isfinite(HP), axis=(-2, -1))
)
print(f'P.shape = {P.shape}（N, K, K） | HP.shape = {HP.shape}（N, K, K）')
print(f'有效 walker 比例 = {float(finite.mean()):.3f}')

P.shape = (3200, 4, 4)（N, K, K） | HP.shape = (3200, 4, 4)（N, K, K）
有效 walker 比例 = 1.000


In [16]:
finite

Array([ True,  True,  True, ...,  True,  True,  True], dtype=bool)

## 2. 从样本估计 $M$、$S$（均为 K×K）再解广义本征问题得 $v$

每 walker 按 Frobenius 范数归一化（P、HP 同乘公共标度，不改变广义本征值）：
$$
\hat S = \frac{1}{N}\sum_n P_n^\dagger P_n,\qquad
\hat M = \frac{1}{N}\sum_n P_n^\dagger (H P)_n
$$
再解 $Mv = \lambda S v$（`scipy.linalg.eigh(M, S)`）：

In [10]:
mask = finite
Pn = jnp.where(mask.reshape(-1, 1, 1), P, 0.0)
HPn = jnp.where(mask.reshape(-1, 1, 1), HP, 0.0)
norm = jnp.sqrt(jnp.sum(jnp.abs(Pn) ** 2, axis=(1, 2)) + 1e-30).reshape(-1, 1, 1)
Pn = Pn / norm
HPn = HPn / norm

N_eff = int(mask.sum())
Shat = jnp.einsum('nαi,nαj->ij', jnp.conj(Pn), Pn).astype(jnp.complex128) / N_eff
Mhat = jnp.einsum('nαi,nαj->ij', jnp.conj(Pn), HPn).astype(jnp.complex128) / N_eff
Shat = 0.5 * (Shat + Shat.conj().T)
Mhat = 0.5 * (Mhat + Mhat.conj().T)

lam, v = eigh(np.asarray(Mhat), np.asarray(Shat))
order = np.argsort(lam.real)
lam, v = lam[order], v[:, order]

print('================ 维度自检（核心） ================')
print(f'M  : {Mhat.shape}   S : {Shat.shape}   v : {v.shape}   （都是 K×K = {K}×{K}）')
print('N_eff (参与统计的有效 walker) =', N_eff)
print('================ 广义本征值 vs FCI ================')
print('广义本征值 λ :', np.round(lam.real, 6))
print('FCI 本征值    :', np.round(E_fcis, 6))
print('误差(Ha)      :', np.round(np.abs(lam.real - np.asarray(E_fcis).real), 6))
print('================ 旋转系数 v（行=能级升序, 列=原始列） ================')
print(np.round(np.real(v), 4))

================ 维度自检（核心） ================
M  : (4, 4)   S : (4, 4)   v : (4, 4)   （都是 K×K = 4×4）
N_eff (参与统计的有效 walker) = 3200
================ 广义本征值 vs FCI ================
广义本征值 λ : [-1.044877 -0.956351 -0.662936 -0.552537]
FCI 本征值    : [-1.054347 -0.957906 -0.668952 -0.551402]
误差(Ha)      : [0.00947  0.001554 0.006016 0.001135]
================ 旋转系数 v（行=能级升序, 列=原始列） ================
[[  1.9494  -0.8355   6.0993   6.6822]
 [ -1.6384  15.831   -1.1686  -7.1967]
 [-26.0344 -10.4208   0.7179 -85.0384]
 [  0.2296   5.856    0.206    1.1968]]


## 3. 与训练估计量的一致性验证

训练把能量当作"平均局部能量矩阵 $\langle P^{-1}HP\rangle$ 的本征值"（即假设列已近似正交、$S\approx I$）。
验证广义本征值 $\lambda$ 与训练轨迹里的能级一致，同时验证每个 walker 的恒等式
$\mathrm{eig}(P^{-1}HP)=\mathrm{gen-eig}(P^\dagger HP,\;P^\dagger P)$。

In [11]:
# (a) 平均局部能量矩阵 <P^{-1} HP> 的本征值（训练的「正交列」近似）
E_L_walk = jnp.linalg.solve(Pn, HPn)                 # (N,K,K) 每 walker 的 E_L 矩阵
E_L_mean = E_L_walk.mean(axis=0)
lam_orth = jnp.linalg.eigvals(E_L_mean)
lam_orth = jnp.sort(lam_orth.real)

# (b) 训练历史里最后保存的能级
train_levels = np.asarray(history['Energy_levels'][-1]).real if 'Energy_levels' in history else None

print('本 notebook 广义本征值 λ   :', np.round(lam.real, 6))
print('平均 <P^{-1}HP> 本征值(正交) :', np.round(lam_orth, 6))
if train_levels is not None:
    print('训练轨迹最后能级            :', np.round(train_levels, 6))
print('FCI 本征值                  :', np.round(E_fcis, 6))

本 notebook 广义本征值 λ   : [-1.044877 -0.956351 -0.662936 -0.552537]
平均 <P^{-1}HP> 本征值(正交) : [-1.046189 -0.956251 -0.660495 -0.546728]
训练轨迹最后能级            : [-1.046189 -0.956251 -0.660495 -0.546728]
FCI 本征值                  : [-1.054347 -0.957906 -0.668952 -0.551402]


In [12]:
# (c) 逐 walker 验证恒等式：eig(P^{-1}HP) == gen-eig(P†HP, P†P)
diff_max = 0.0
for m in range(min(N, 50)):
    Pm = Pn[m]; HPm = HPn[m]
    e1 = np.sort(np.linalg.eigvals(np.linalg.solve(Pm, HPm)).real)
    Ag = np.asarray(Pm.conj().T @ HPm)
    Sg = np.asarray(Pm.conj().T @ Pm)
    e2 = np.sort(np.linalg.eigvals(np.linalg.solve(Sg, Ag)).real)
    diff_max = max(diff_max, np.max(np.abs(e1 - e2)))
print(f'在 {min(N,50)} 个 walker 上二者本征值最大偏差 = {diff_max:.2e}（都 <1e-12 → 恒等式成立）')

在 50 个 walker 上二者本征值最大偏差 = 2.89e-15（都 <1e-12 → 恒等式成立）


## 4. 结论

- 重建 $M$, $S$, $v$ **不需要**枚举全部构型，也不需要完整 16×16 的 H；只用已采样的 walker 构型 + 局部能量算子，**所有中间矩阵都是 K×K**。
- 计算 $v$ 只解一个 **K×K=4×4 的广义本征问题** `eigh(M, S)`，成本与"精确对角化 16×16 的 H"**完全无关**，不会退化。
- 与训练一致：训练把能级取作 $\langle P^{-1}HP\rangle$ 的本征值（列近似正交）；广义本征 $Mv=\lambda Sv$ 是其在列不正交时更严格的版本，本 walker 恒等式 $\mathrm{eig}(P^{-1}HP)=\mathrm{gen\text{-}eig}(P^\dagger HP,P^\dagger P)$ 逐点成立。